# NB01 — Environment Setup & Compute Passport

Sets up paths, logs the environment, probes one WSI to verify OpenSlide is working, and writes the compute passport that subsequent notebooks append to.

**Paper:** Zafar SA, Qin W, Liu C, Khan AA, Nazir A, Khalid F, Faisal MS. *OpenSlideFM: A Computationally Efficient Multi-Scale Foundation Model for Computational Pathology* (2026).

Set `WORKSPACE` and `WSI_ROOT` environment variables before running, or run from a directory containing the data folders directly.

In [ ]:
import os, sys, json, platform, shutil, socket, datetime
from pathlib import Path
from typing import Any, Dict

WORKSPACE = Path(os.environ.get('WORKSPACE', './workspace'))
WSI_ROOT  = Path(os.environ.get('WSI_ROOT',  './data/wsi'))
PANDA_ROOT = Path(os.environ.get('PANDA_ROOT', './data/PANDA'))

SUBDIRS = {
    'compute':     WORKSPACE / 'compute',
    'logs':        WORKSPACE / 'logs',
    'figures':     WORKSPACE / 'figures',
    'qc':          WORKSPACE / 'qc',
    'tiles':       WORKSPACE / 'tiles',
    'features':    WORKSPACE / 'features',
    'embeddings':  WORKSPACE / 'embeddings',
    'attn':        WORKSPACE / 'attn',
    'leak_audit':  WORKSPACE / 'leak_audit',
    'preanalytics': WORKSPACE / 'preanalytics',
    'artifacts':   WORKSPACE / 'artifacts',
    'manifests':   WORKSPACE / 'manifests',
    'hashes':      WORKSPACE / 'hashes',
    'ckpt':        WORKSPACE / 'ckpt',
    'weights':     WORKSPACE / 'weights',
    'results':     WORKSPACE / 'results',
    'diagnostics': WORKSPACE / 'diagnostics',
}
for p in SUBDIRS.values():
    p.mkdir(parents=True, exist_ok=True)

# optional imports that may not be available on every machine
OPENS = None; PIL_Image = None; TORCH = None
try:
    import openslide as OPENS
except Exception as e:
    print('[WARN] openslide-python not available:', e)
try:
    from PIL import Image as PIL_Image
except Exception as e:
    print('[WARN] Pillow not available:', e)
try:
    import torch as TORCH
except Exception as e:
    print('[WARN] PyTorch not available:', e)

def now_iso() -> str:
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def gb(nbytes: int) -> float:
    return round(nbytes / (1024**3), 2)

def safe_write_json(path: Path, obj: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.parent / (path.name + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    tmp.replace(path)

def list_wsi_files(root: Path):
    if not root.exists():
        return []
    exts = ('.svs', '.tif', '.tiff', '.ndpi', '.mrxs', '.scn')
    files = []
    for ext in exts:
        files.extend(root.rglob(f'*{ext}'))
        files.extend(root.rglob(f'*{ext.upper()}'))
    return sorted(set(files))

def open_and_probe_wsi(path: Path):
    if OPENS is None:
        return None
    slide = OPENS.OpenSlide(str(path))
    props = slide.properties
    w, h = slide.dimensions
    mpp_x = props.get('openslide.mpp-x') or props.get('aperio.MPP') or None
    mpp_y = props.get('openslide.mpp-y') or props.get('aperio.MPP') or None
    vendor = props.get('openslide.vendor') or props.get('aperio.AppMag') or 'unknown'
    thumb_path = None
    try:
        if hasattr(slide, 'get_thumbnail') and PIL_Image is not None:
            max_side = 768
            scale = max(w, h) / max_side if max(w, h) > max_side else 1.0
            tw, th = int(w/scale), int(h/scale)
            thumb = slide.get_thumbnail((tw, th))
            thumb_path = SUBDIRS['figures'] / f'sample_thumb_{path.stem}.jpg'
            thumb.save(str(thumb_path), 'JPEG', quality=90)
    except Exception as e:
        print(f'[WARN] could not write thumbnail for {path.name}: {e}')
        thumb_path = None
    finally:
        slide.close()
    return (w, h, mpp_x, mpp_y, vendor, thumb_path)

def get_env_summary() -> Dict[str, Any]:
    info = {
        'timestamp': now_iso(),
        'host': socket.gethostname(),
        'platform': platform.platform(),
        'python': sys.version.replace('\n', ' '),
        'workspace': str(WORKSPACE),
        'wsi_root': str(WSI_ROOT),
        'paths_note': "all writes go under WORKSPACE; WSI_ROOT is read-only",
    }
    try:
        total, used, free = shutil.disk_usage(WORKSPACE)
        info.update({'disk_total_gb': gb(total), 'disk_used_gb': gb(used), 'disk_free_gb': gb(free)})
    except Exception as e:
        info['disk_error'] = str(e)
    if TORCH is not None:
        info['torch_version'] = TORCH.__version__
        info['cuda_available'] = TORCH.cuda.is_available()
        if TORCH.cuda.is_available():
            try:
                dev = TORCH.cuda.current_device()
                prop = TORCH.cuda.get_device_properties(dev)
                info['cuda_device'] = {
                    'index': dev, 'name': prop.name,
                    'total_vram_gb': round(prop.total_memory/(1024**3), 2),
                    'multi_processor_count': getattr(prop, 'multi_processor_count', None),
                }
                info['cuda_runtime_version'] = TORCH.version.cuda
                info['cudnn_version'] = TORCH.backends.cudnn.version()
            except Exception as e:
                info['cuda_error'] = str(e)
    else:
        info['torch_version'] = None
    info['openslide_version'] = getattr(OPENS, '__version__', None) if OPENS else None
    return info

print(f'[{now_iso()}] workspace: {WORKSPACE}')
print(f'[{now_iso()}] WSI root (read-only): {WSI_ROOT}')
if not WSI_ROOT.exists():
    print(f'[WARN] WSI_ROOT does not exist yet: {WSI_ROOT}')

env = get_env_summary()
compute_passport = {
    'run_id': datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
    'created_at': now_iso(),
    'workspace': str(WORKSPACE),
    'wsi_root': str(WSI_ROOT),
    'environment': env,
    'stages': [],
}
compute_path = SUBDIRS['compute'] / 'compute_passport.json'
safe_write_json(compute_path, compute_passport)
print(f'[OK] compute passport at: {compute_path}')

wsi_files = list_wsi_files(WSI_ROOT)
print(f'[INFO] detected {len(wsi_files)} WSI files in WSI_ROOT')

sample_report = {}
if wsi_files and OPENS is not None:
    sample_path = wsi_files[0]
    try:
        probe = open_and_probe_wsi(sample_path)
        if probe:
            w, h, mpp_x, mpp_y, vendor, thumb_path = probe
            sample_report = {
                'slide_path': str(sample_path), 'width': w, 'height': h,
                'mpp_x': mpp_x, 'mpp_y': mpp_y, 'vendor': vendor,
                'thumbnail': str(thumb_path) if thumb_path else None,
            }
            print(f'  sample slide: {sample_report["slide_path"]}')
            print(f'  size: {w} x {h}')
            print(f'  mpp: x={mpp_x} y={mpp_y}')
            print(f'  vendor: {vendor}')
    except Exception as e:
        print(f'[WARN] could not open sample slide: {e}')
else:
    if not wsi_files:
        print('[INFO] no WSI files detected')
    if OPENS is None:
        print('[WARN] openslide-python missing; install with: pip install openslide-python openslide-bin')

log_path = SUBDIRS['logs'] / 'env_summary.json'
safe_write_json(log_path, {'timestamp': now_iso(), 'env': env, 'sample_probe': sample_report})
print(f'[OK] environment summary: {log_path}')

txt_path = SUBDIRS['logs'] / 'env_summary.txt'
with txt_path.open('w', encoding='utf-8') as f:
    f.write('OpenSlideFM environment summary\n')
    f.write(f'Timestamp: {now_iso()}\n\n')
    for k, v in env.items():
        f.write(f'{k}: {v}\n')
    if sample_report:
        f.write('\nSample WSI probe:\n')
        for k, v in sample_report.items():
            f.write(f'  {k}: {v}\n')

print('NB01 complete. Proceed to NB02 (manifest & provenance).')